# Công đoạn 05 — Lựa chọn đặc trưng F54
## 1. Mục tiêu
Nghiên cứu phương sai, tương quan, nội dung và minh họa lựa chọn đặc trưng chỉ trên Train. Giữ baseline đủ 54 câu theo thứ tự gốc; không tính RIASEC khi chưa có codebook, không sửa nội dung và không mặc định ít câu hơn tốt hơn. Chưa huấn luyện classifier hoặc đánh giá Accuracy, Top-K, F1.

## 2. Cơ sở lý thuyết
VarianceThreshold(0) chỉ loại cột hằng, không sử dụng y; phương sai phụ thuộc thang đo. Spearman mô tả quan hệ đơn điệu giữa biến thứ bậc, không tự chứng minh dư thừa nội dung.

SelectKBest(f_classif) xếp hạng từng biến bằng thống kê F ANOVA giữa các lớp. Phương pháp đơn biến không xét tương tác hoặc dư thừa giữa câu. Likert là thứ bậc; khoảng cách số, giả định phân phối/phương sai và cỡ mẫu nhỏ của 33 lớp hạn chế cách diễn giải F/p-value. Các điểm này không kiểm định giá trị tâm lý hoặc tầm quan trọng Holland; không dùng p-value để kết luận sau nhiều phép thử.

## 3. Tái lập Train
Tái hiện làm sạch Notebook 01: giữ Likert văn bản hợp lệ, trim nhãn; không xóa mẫu. Giữ thứ tự 54 câu. Tái lập grouping tuple và StratifiedGroupKFold(5, shuffle=True, random_state=42), fold đầu tiên. Đối chiếu trực tiếp code phép chia Notebook 02 và index; chỉ đọc notebook đó để kiểm chứng, không dùng biến kernel cũ. Không thực hiện EDA Test.

In [1]:
from pathlib import Path
import sys, ast
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
import nbformat
assert sys.version_info[:2] == (3,14)
ROOT = next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'data_AI_1000_rows.xlsx').is_file())
raw = pd.read_excel(ROOT/'data_AI_1000_rows.xlsx',sheet_name='AUGMENTED_RAW_1000')
metadata = ['Dấu thời gian','Tên Sinh viên','Ngành học','Giới tính']
assert raw.shape == (1000,58) and list(raw.columns[:4]) == metadata
questions = list(raw.columns[4:])
levels = ['Chưa bao giờ đúng','Chỉ đúng 1 vài trường hợp','Chỉ đúng 1 nửa','Gần như là đúng','Hoàn toàn đúng']
X = raw[questions].copy()
y = raw['Ngành học'].astype('string').str.strip()
assert X.notna().all().all() and X.isin(levels).all().all()
assert y.notna().all() and y.ne('').all() and y.nunique() == 33
assert X.shape[1] == 54 and not set(metadata).intersection(X.columns)
groups = pd.Series(pd.factorize(pd.Series(list(X.itertuples(index=False, name=None))), sort=False)[0], index=X.index)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_pos, test_pos = next(splitter.split(X, y, groups))
ref = nbformat.read(ROOT/'data/ml_pipeline/notebooks/02_chia_du_lieu.ipynb',4)
src = next(c.source for c in ref.cells if c.cell_type=='code' and 'from sklearn.model_selection import StratifiedGroupKFold' in c.source)
nodes = [node for node in ast.parse(src).body if isinstance(node,ast.Assign) and any(
    isinstance(t,ast.Name) and t.id in {'groups','splitter'} or isinstance(t,ast.Tuple) and [v.id for v in t.elts]==['train_pos','test_pos'] for t in node.targets)]
assert len(nodes)==3
check = {'pd':pd,'X':X,'y':y,'StratifiedGroupKFold':StratifiedGroupKFold}
exec(compile(ast.Module(body=nodes,type_ignores=[]),'<Notebook02 split>','exec'),check)
assert groups.equals(check['groups'])
assert np.array_equal(train_pos,check['train_pos']) and np.array_equal(test_pos,check['test_pos'])
X_train,y_train = X.iloc[train_pos].copy(),y.iloc[train_pos].copy()
assert X_train.shape==(800,54) and y_train.nunique()==33
assert len(test_pos)==200 and len(train_pos)+len(test_pos)==len(X)
assert X.index.is_unique and not set(train_pos).intersection(test_pos)
assert set(groups.iloc[train_pos]).isdisjoint(groups.iloc[test_pos])
print('PASS: index đúng Notebook 02; Train 800 x 54, 33 ngành; Test 200; không giao index/vector.')
groups_train = groups.iloc[train_pos].copy()
del raw,X,y,groups,check,test_pos,ref


PASS: index đúng Notebook 02; Train 800 x 54, 33 ngành; Test 200; không giao index/vector.


## 4. Phương án A và B — Đủ 54 câu, phương sai
Mã hóa cố định năm nhãn thành 1–5 trên Train, không học tham số. Dùng phương sai ddof=0; không tìm ngưỡng để ép giảm số câu. VarianceThreshold dưới đây chỉ minh họa thống kê toàn Train; khi đánh giá CV phải fit lại bên trong Pipeline.

In [2]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
X_train_numeric = X_train.apply(lambda s:s.map(dict(zip(levels,range(1,6))))).astype('int64')
X_train_numeric.columns = [f'q{i}' for i in range(1,55)]
assert X_train_numeric.shape == (800,54) and X_train_numeric.isin(range(1,6)).all().all()
content = pd.Series(questions,index=X_train_numeric.columns,name='Nội dung')
X_baseline = X_train_numeric.copy()
assert list(X_baseline.columns) == [f'q{i}' for i in range(1,55)]
variance = X_train_numeric.var(ddof=0).rename('Variance').to_frame()
zero = variance.index[variance['Variance'].eq(0)].tolist()
variance_selector_demo = VarianceThreshold(threshold=0)
variance_selector_demo.fit(X_train_numeric)
kept = X_train_numeric.columns[variance_selector_demo.get_support()].tolist()
assert np.allclose(variance_selector_demo.variances_,variance['Variance'])
display(variance.round(6))
print('Cột hằng:',zero,'; số câu giữ:',len(kept))

Cột hằng: [] ; số câu giữ: 54


     Variance
q1   1.063944
q2   0.961986
q3   1.051900
q4   1.073494
q5   1.356400
q6   1.194400
q7   1.117623
q8   1.309600
q9   1.252600
q10  1.181194
q11  1.020111
q12  1.041436
q13  1.544944
q14  1.184400
q15  1.513975
q16  1.183086
q17  1.311744
q18  0.997344
q19  1.426194
q20  1.285123
q21  1.071875
q22  1.333400
q23  1.503694
q24  1.649475
q25  1.406744
q26  1.332994
q27  1.269944
q28  0.782173
q29  0.891861
q30  0.673275
q31  1.418975
q32  0.793311
q33  1.297994
q34  1.216211
q35  0.797275
q36  1.225944
q37  1.532686
q38  1.198048
q39  1.249475
q40  1.235275
q41  1.528361
q42  1.214811
q43  1.460486
q44  1.526798
q45  1.534844
q46  1.180494
q47  1.194561
q48  1.057475
q49  1.613975
q50  1.489648
q51  1.125775
q52  1.342048
q53  1.372100
q54  1.639298

## 5. Tương quan và nội dung q42–q51
Báo cáo cả hai ngưỡng |rho| ≥0,8 và ≥0,85; không tự loại câu theo tương quan. So sánh với Notebook 03: không có cặp ≥0,8, q42–q51 khoảng 0,579.

In [3]:
corr = X_train_numeric.corr(method='spearman')
pairs = pd.DataFrame([(corr.columns[i],corr.columns[j],corr.iloc[i,j]) for i in range(54) for j in range(i+1,54)],columns=['Câu 1','Câu 2','rho'])
pairs['|rho|'] = pairs['rho'].abs()
for threshold in [.8,.85]:
    result = pairs[pairs['|rho|']>=threshold]
    print('Ngưỡng:',threshold,'; số cặp:',len(result))
    display(result)
print('q42:',content['q42'])
print('q51:',content['q51'])
rho = corr.loc['q42','q51']
print('Spearman q42–q51:',rho)
assert (pairs['|rho|']>=.8).sum()==0
assert np.isclose(rho,.5787159815402011)
print('Kết quả khớp thống kê Notebook 03.')

Ngưỡng: 0.8 ; số cặp: 0
Ngưỡng: 0.85 ; số cặp: 0
q42: 42. Tôi thường đặt ra các mục tiêu, kế hoạch trong cuộc sống
q51: 51. Tôi thường đặt ra mục tiêu, kế hoạch trong cuộc sống
Spearman q42–q51: 0.5787159815402011
Kết quả khớp thống kê Notebook 03.


Empty DataFrame
Columns: [Câu 1, Câu 2, rho, |rho|]
Index: []

Empty DataFrame
Columns: [Câu 1, Câu 2, rho, |rho|]
Index: []

Nhận xét: q42 “Tôi thường đặt ra các mục tiêu, kế hoạch trong cuộc sống” và q51 “Tôi thường đặt ra mục tiêu, kế hoạch trong cuộc sống” gần như cùng nội dung, chỉ khác từ “các”. Tương quan khoảng 0,579 chưa đạt ngưỡng cao quy ước. Nội dung tương tự là lý do cần thẩm định, không phải căn cứ khoa học đủ để chọn bỏ một câu. Thiếu codebook, đánh giá tâm trắc và bằng chứng độ tin cậy/giá trị đo lường; giữ cả hai trong baseline. Không sửa câu khảo sát.

## 6. SelectKBest trong một fold minh họa
Tạo SGKF 5 folds trên Train, shuffle=True, random_state=42; lấy fold đầu tiên cố định. Group tiếp tục dựa trên đủ 54 đáp án, không tạo lại theo đặc trưng đã chọn. Chỉ fit selector trên phần training của fold; không fit validation hoặc Test. Minh họa k=20,35,54 là ứng viên định trước, không phải tối ưu. Không dùng danh sách của fold này làm tập đặc trưng cuối cùng hoặc đánh giá CV sau đó.

In [4]:
inner = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
inner_train_pos, inner_valid_pos = next(inner.split(X_train_numeric,y_train,groups_train))
X_fold_train = X_train_numeric.iloc[inner_train_pos].copy()
y_fold_train = y_train.iloc[inner_train_pos].copy()
assert set(inner_train_pos).isdisjoint(inner_valid_pos)
assert len(inner_train_pos)+len(inner_valid_pos)==800
assert set(groups_train.iloc[inner_train_pos]).isdisjoint(groups_train.iloc[inner_valid_pos])
assert set(y_fold_train)==set(y_train)
assert set(y_train.iloc[inner_valid_pos])==set(y_train)
assert groups_train.index.equals(X_train_numeric.index)
print('Fold minh họa training/validation:',len(inner_train_pos),len(inner_valid_pos))
print('Số ngành training/validation:',y_fold_train.nunique(),y_train.iloc[inner_valid_pos].nunique())
selected_by_k = {}
for k in [20,35,54]:
    selector_demo = SelectKBest(score_func=f_classif,k=k)
    selector_demo.fit(X_fold_train,y_fold_train)
    assert selector_demo.n_features_in_==54
    selected_by_k[k] = X_fold_train.columns[selector_demo.get_support()].tolist()
    assert len(selected_by_k[k])==k
    print('k =',k,':',', '.join(selected_by_k[k]))
# F/p-value giống nhau giữa các k vì cùng dữ liệu fit; k chỉ đổi support.
scores = pd.DataFrame({'Câu':X_fold_train.columns,'F':selector_demo.scores_,
    'p-value (chưa hiệu chỉnh)':selector_demo.pvalues_})
scores['Nội dung'] = scores['Câu'].map(content)
scores['Chọn k=20'] = scores['Câu'].isin(selected_by_k[20])
scores['Chọn k=35'] = scores['Câu'].isin(selected_by_k[35])
display(scores.sort_values('F',ascending=False).round(6))
assert np.isfinite(scores['F']).all()
print('PASS: selector chỉ fit X_fold_train/y_fold_train; không fit validation/Test.')

Fold minh họa training/validation: 639 161
Số ngành training/validation: 33 33
k = 20 : q4, q5, q7, q11, q12, q13, q15, q17, q18, q19, q21, q22, q25, q33, q36, q39, q40, q41, q50, q54
k = 35 : q1, q4, q5, q7, q9, q10, q11, q12, q13, q15, q16, q17, q18, q19, q20, q21, q22, q23, q24, q25, q26, q33, q34, q35, q36, q38, q39, q40, q41, q44, q46, q48, q49, q50, q54
k = 54 : q1, q2, q3, q4, q5, q6, q7, q8, q9, q10, q11, q12, q13, q14, q15, q16, q17, q18, q19, q20, q21, q22, q23, q24, q25, q26, q27, q28, q29, q30, q31, q32, q33, q34, q35, q36, q37, q38, q39, q40, q41, q42, q43, q44, q45, q46, q47, q48, q49, q50, q51, q52, q53, q54
PASS: selector chỉ fit X_fold_train/y_fold_train; không fit validation/Test.


    Câu         F  p-value (chưa hiệu chỉnh)                                                                       Nội dung  Chọn k=20  Chọn k=35
10  q11  3.182649                   0.000000                                           11. Tôi có khả năng phân tích vấn đề       True       True
39  q40  2.965041                   0.000000             40. Tôi có khả năng diễn đạt, tranh luận và thuyết phục người khác       True       True
21  q22  2.904291                   0.000000                                    22. Tôi có khả năng thuyết trình, diễn xuất       True       True
12  q13  2.866946                   0.000000                          13. Tôi thích thực hiện các thí nghiệm hay nghiên cứu       True       True
3    q4  2.845116                   0.000001                        4. Tôi có thể vận hành, điều khiển các máy móc thiết bị       True       True
24  q25  2.761099                   0.000001                    25. Tôi có khả năng viết , trình bày những ý tưởng của mình 

Nhận xét: bảng điểm và danh sách trên chỉ thuộc một training fold. Việc chọn câu có thể thay đổi giữa folds; chưa kiểm tra độ ổn định. F cao chỉ thể hiện khác biệt nhóm trong mẫu theo phép tính ANOVA; không đồng nghĩa quan trọng nhất cho tư vấn nghề nghiệp.

## 7. Phương án đề xuất Notebook 06
| Phương án | Nội dung | Vai trò và giới hạn |
|---|---|---|
| A | Đủ 54 câu | Baseline bắt buộc so sánh; giữ q42 và q51 |
| B | VarianceThreshold(0) trong Pipeline | Không giảm chiều trên Train hiện tại; có thể làm kiểm tra cột hằng trong từng fold, không bắt buộc một thử nghiệm riêng nếu support giống A |
| C | SelectKBest(f_classif,k=20) trong Pipeline | Ứng viên giảm chiều, chưa xác nhận tốt hơn |
| D | SelectKBest(f_classif,k=35) trong Pipeline | Ứng viên giảm chiều, chưa xác nhận tối ưu |

k=54 trong minh họa giữ đủ câu như A về support, không cần thêm một thử nghiệm riêng chỉ vì có selector.

## 8. Nguyên tắc Pipeline và chống leakage
Notebook 06 phải dùng sklearn Pipeline gồm scaler nếu thuật toán cần, feature selector nếu có và classifier. Mọi thành phần có fit phải fit lại riêng trên training portion của từng fold. Không đưa ma trận chuẩn hóa toàn Train từ Notebook 04 hoặc danh sách câu của fold minh họa này vào CV như lựa chọn cố định đã học trước. Nếu dùng VarianceThreshold(0), đặt trước scaler để kiểm tra cột hằng trên dữ liệu gốc của fold. Giữ group từ đủ 54 câu xuyên suốt CV; không tạo group sau chọn câu.

Chọn mô hình/k bằng CV trên Train; không dùng Test để lựa chọn. Sau khi chốt cấu hình mới fit Pipeline cuối trên toàn Train và đánh giá Test một lần. Notebook này không xây dựng classifier hoặc tính hiệu năng.


## 9. Kết luận và kiểm chứng
- Có 0 đặc trưng phương sai bằng 0; VarianceThreshold(0) giữ 54/54 câu.
- Không có cặp |Spearman| ≥0,8 hoặc ≥0,85, phù hợp Notebook 03.
- q42–q51 có rho=0.578716, nội dung gần trùng; giữ cả hai trong baseline và cần thẩm định khoa học trước quyết định.
- Danh sách k=20,35,54 được in đầy đủ ở phần 6, chỉ là minh họa trên training của fold đầu tiên, không phải danh sách cuối cùng.
- Chưa có codebook RIASEC hoặc xác thực tâm trắc; không thể suy ra tầm quan trọng Holland từ F hoặc nhãn ngành đang học.
- Đề xuất so sánh baseline 54 câu với k=20 và k=35 trong Pipeline; VarianceThreshold(0) là kiểm tra cột hằng, không ép giảm chiều. Chưa xác định số câu tối ưu.

Đã chạy tuần tự 4/4 code cell trong tiến trình Python 3.14 mới; outputs lưu trong notebook. Dùng phương án này theo lỗi quyền Jupyter Windows đã ghi nhận; chưa kiểm chứng bằng kernel Jupyter. Outer split được đối chiếu index với Notebook 02; inner fold tách index/group. Test chỉ tham gia kiểm tra phép chia, không dùng nghiên cứu feature selection. Không fit SelectKBest trên toàn 800 mẫu, validation hoặc Test. Không huấn luyện, tính Accuracy/Top-K/F1 hay xuất file phụ. Dừng ở Notebook 05.
